Take these two algorithms, PSO and ABC. Propose a modification to the basic algorithm. The modification must be a significant change to the behavior of the algorithm. A simple randomization or change in parameter values is not considered significant.

Compare the performance of these algorithms and their modificaiton on the following functions:
a.	Rastrigin Function
b.	Styblinski-Tang Function

In [12]:
import numpy as np
import random
import math

In [13]:
# PSO parameters
MAX_VELOCITY = 1.0  # Maximum velocity for particles
BOUNDS = [-5, 5]    # Search space bounds for x and y

class Particle:
    def __init__(self, position):
        self.position =   position
        self.velocity =   [random.uniform(-MAX_VELOCITY, MAX_VELOCITY) for _ in range(2)]  # Initial random velocity
        self.p_best =     position  # Best position found by this particle
        self.p_best_fitness =   self.fitness()  # Fitness of the best position

    def fitness(self):
        # Calculate the value of the function f(x, y) = x^2 + y^2
        return self.position[0]**2 + self.position[1]**2


    def update(self, g_best, c1=1, c2=1):
        for i in range(2):
            # Update velocity
            self.velocity[i] += c1 * random.random() * (self.p_best[i] - self.position[i]) + c2 * random.random() * (g_best[i] - self.position[i])
            
            # Limit velocity to avoid explosion using numpy.clip
            self.velocity[i] = np.clip(self.velocity[i], -MAX_VELOCITY, MAX_VELOCITY)

            # Update position
            self.position[i] += self.velocity[i]

            # Constrain position to search space bounds using numpy.clip
            self.position[i] = np.clip(self.position[i], BOUNDS[0], BOUNDS[1])


        # Update personal best if the current position is better
        current_fitness = self.fitness()
        if current_fitness < self.p_best_fitness:
            self.p_best = self.position.copy()
            self.p_best_fitness = current_fitness


class ModifiedParticle(Particle):
    def __init__(self, position, stagnation_limit=5):
        super().__init__(position)
        self.stagnation = 0
        self.stagnation_limit = stagnation_limit

    def update(self, g_best, c1=1, c2=1):
        prev_best = self.p_best_fitness
        super().update(g_best, c1, c2)

        if self.p_best_fitness < prev_best:
            self.stagnation = 0
        else:
            self.stagnation += 1

        if self.stagnation > self.stagnation_limit:
            self.reinitialize()

    def reinitialize(self):
        self.position = [random.uniform(BOUNDS[0], BOUNDS[1]) for _ in range(2)]
        self.velocity = [random.uniform(-MAX_VELOCITY, MAX_VELOCITY) for _ in range(2)]
        self.p_best = self.position.copy()
        self.p_best_fitness = self.fitness()
        self.stagnation = 0
        

class Swarm:
    def __init__(self, num_particles, objective_fn, is_minimization=True):
        # Initialize particles with random positions
        self.swarm = []
        self.objective_fn = objective_fn
        self.is_minimization = is_minimization
        for _ in range(num_particles):
            position = [random.uniform(BOUNDS[0], BOUNDS[1]), random.uniform(BOUNDS[0], BOUNDS[1])]
            self.swarm.append(Particle(position))


    def find_global_best(self):
        # Find the particle with the best fitness
        if self.is_minimization:
            return min((p.p_best for p in self.swarm), key=lambda x: sum(xi ** 2 for xi in x))
        else:
            return max((p.p_best for p in self.swarm), key=lambda x: sum(xi ** 2 for xi in x))

    def fitness(self, position):
        # Calculate the value of the objective function at the given position
        return self.objective_fn(position)

    def update(self):
        for particle in self.swarm:
            particle.update(self.g_best)


    def simulate(self):
        """Generator function to yield the swarm state at each step."""
        while True:
            self.g_best = self.find_global_best()
            self.update()
            yield self.swarm, self.g_best


class ModifiedSwarm(Swarm):
    def __init__(self, num_particles, objective_fn, is_minimization=True, stagnation_limit=5):
        self.stagnation_limit = stagnation_limit
        super().__init__(num_particles, objective_fn, is_minimization)
        # replace particles with modified ones while keeping existing best via p_best
        self.swarm = [ModifiedParticle(p.position, stagnation_limit) for p in self.swarm]

    def update(self):
        for particle in self.swarm:
            particle.update(self.g_best)

In [14]:
class ArtificialBeeColony:
    def __init__(self, objective_function, num_food_sources=20, num_dimensions=2, max_iterations=100, limit=10, bounds=(-5, 5), is_minimization=True):
        """
        Initialize the ABC algorithm.

        Parameters:
            objective_function (callable): The objective function to minimize.
            num_food_sources (int): Number of food sources (candidate solutions).
            num_dimensions (int): Number of dimensions in the search space.
            max_iterations (int): Maximum number of iterations.
            limit (int): Abandonment limit for scout bees.
            bounds (tuple): Lower and upper bounds for the search space.
        """
        self.objective_function = objective_function
        self.num_food_sources = num_food_sources
        self.num_dimensions = num_dimensions
        self.max_iterations = max_iterations
        self.limit = limit
        self.bounds = bounds
        self.is_minimization = is_minimization

        # Initialize food sources (candidate solutions)
        self.food_sources = np.random.uniform(self.bounds[0], self.bounds[1], (self.num_food_sources, self.num_dimensions))
        self.fitness_values = np.zeros(self.num_food_sources)
        self.trials = np.zeros(self.num_food_sources)
        self.best_solution = None
        self.best_fitness = float('inf')

    def fitness(self, f):
        """
        Calculate the fitness of a solution.

        Parameters:
            f (float): Objective function value.

        Returns:
            float: Fitness value.
        """
        return 1 / (1 + f) if f >= 0 else 1 + abs(f)

    def evaluate_fitness(self):
        """
        Evaluate the fitness of all food sources.
        """
        if self.is_minimization:
            self.best_fitness = float('inf')
        else:
            self.best_fitness = float('-inf')
        for i in range(self.num_food_sources):
            f_x = self.objective_function(self.food_sources[i])
            self.fitness_values[i] = self.fitness(f_x)
            if (self.is_minimization and f_x < self.best_fitness) or (not self.is_minimization and f_x > self.best_fitness):
                self.best_fitness = f_x
                self.best_solution = self.food_sources[i]

    def employed_bee_phase(self):
        """
        Employed bee phase: Explore new solutions around existing food sources.
        """
        for i in range(self.num_food_sources):
            # Select a random neighbor
            rand_neighbor = np.random.choice([j for j in range(self.num_food_sources) if j != i])
            # Select a random dimension
            rand_dim = np.random.randint(self.num_dimensions)

            # Generate a new solution
            solution = np.copy(self.food_sources[i])
            phi = np.random.uniform(-1, 1)
            # 𝑋_𝑛𝑒𝑤 = 𝑋 + 𝜙 * (𝑋 − 𝑋_𝑝 )
            solution[rand_dim] = self.food_sources[i][rand_dim] + phi * (self.food_sources[i][rand_dim] - self.food_sources[rand_neighbor][rand_dim])

            # Clip the solution to stay within bounds
            solution = np.clip(solution, self.bounds[0], self.bounds[1])
           
            # Evaluate the new solution
            # calculate the objective function value for the new solution
            f_x = self.objective_function(solution)
            # calculate the fitness for the new solution
            new_fitness = self.fitness(f_x)
            if new_fitness > self.fitness_values[i]:  # If the new solution is better
                self.food_sources[i] = solution
                self.fitness_values[i] = new_fitness
                self.trials[i] = 0  # Reset trials for this food source
            else:
                self.trials[i] += 1  # Increment trials for this food source
            

    def onlooker_bee_phase(self):
        """
        Onlooker bee phase: Select food sources based on fitness and explore around them.
        """
        probabilities = self.fitness_values / np.sum(self.fitness_values)
        for i in range(self.num_food_sources):
            if np.random.rand() < probabilities[i]:
                # Perform the same steps as the employed bee phase
                self.employed_bee_phase()

    def scout_bee_phase(self):
        """
        Scout bee phase: Replace abandoned food sources with new random solutions.
        """
        for i in range(self.num_food_sources):
            if self.trials[i] > self.limit:
                self.food_sources[i] = np.random.uniform(self.bounds[0], self.bounds[1], self.num_dimensions)
                self.trials[i] = 0  # Reset trials for this food source

    def optimize(self):
        """
        Run the ABC optimization algorithm.

        Returns:
            tuple: Best solution and best fitness value.
        """
        self.evaluate_fitness()  # Evaluate initial fitness

        for _ in range(self.max_iterations):
            self.employed_bee_phase()
            self.onlooker_bee_phase()
            self.scout_bee_phase()
            self.evaluate_fitness()  # Update best solution

        return self.best_solution, self.best_fitness

In [15]:
def Rastrigin_function(x):
    """Gobal minimum at x = (1, 1, ..., 1) with f(x) = 0."""
    return 10 * len(x) + sum([(xi - 1) ** 2 - 10 * math.cos(2 * math.pi * (xi - 1)) for xi in x])

def Styblinski_Tang_function(x):
    """Global minimum at x = (-2.903534, -2.903534) with f(x) = -39.16617."""
    return 0.5 * sum([xi ** 4 - 16 * xi ** 2 + 5 * xi for xi in x])

In [16]:
# Baseline test with unmodified PSO and ABC algorithms
if __name__ == "__main__":
    # Test PSO
    print("Testing PSO with Rastrigin function")
    swarm = Swarm(num_particles=30, objective_fn=Rastrigin_function)
    pso_simulation = swarm.simulate()
    for _ in range(100):  # Run for 100 iterations
        swarm_state, g_best = next(pso_simulation)
    print(f"PSO Best Solution: {g_best}, Fitness: {swarm.fitness(g_best)}")

    # Test ABC
    print("\nTesting ABC with Rastrigin function")
    abc = ArtificialBeeColony(objective_function=Rastrigin_function, num_food_sources=30, max_iterations=100)
    best_solution, best_fitness = abc.optimize()
    print(f"ABC Best Solution: {best_solution}, Fitness: {best_fitness}")

    # Test PSO with Styblinski-Tang function
    print("\nTesting PSO with Styblinski-Tang function")
    swarm = Swarm(num_particles=30, objective_fn=Styblinski_Tang_function)
    pso_simulation = swarm.simulate()
    for _ in range(100):  # Run for 100 iterations
        swarm_state, g_best = next(pso_simulation)
    print(f"PSO Best Solution: {g_best}, Fitness: {swarm.fitness(g_best)}")

    # Test ABC with Styblinski-Tang function
    print("\nTesting ABC with Styblinski-Tang function")
    abc = ArtificialBeeColony(objective_function=Styblinski_Tang_function, num_food_sources=30, max_iterations=100)
    best_solution, best_fitness = abc.optimize()
    print(f"ABC Best Solution: {best_solution}, Fitness: {best_fitness}")
    

Testing PSO with Rastrigin function
PSO Best Solution: [0.0007219599874185301, -0.012961636221687056], Fitness: 2.0578951011753546

Testing ABC with Rastrigin function
ABC Best Solution: [0.99602003 1.00778507], Fitness: 0.015164031229602415

Testing PSO with Styblinski-Tang function
PSO Best Solution: [0.01513584702687263, -0.016273697158477707], Fitness: -0.006795980694110033

Testing ABC with Styblinski-Tang function
ABC Best Solution: [-2.9014997  -2.90155026], Fitness: -78.33219189311576


1.	Explain why you made those modifications and how it may improve the optimization.

**PSO Modification**
- The Particle was modified to be randomly initialized, when stagnation is detected, this should ensure that the particle is not stuck in a local minimum and encourages to explore further.

2.	For each of the swarm algorithm (2 plus 2 modified), compute the optimization results on the Rastrigin and ST function for variables 5, 10, 50 for a range of iterations and population (your choice). Please use higher number of iterations and population for high dimensions.

In [19]:
# Result using modified PSO algorithm
iterations = [5, 10, 50]

if __name__ == "__main__":
    # Test modified PSO
    for iters in iterations:
        print(f"\nTesting Modified PSO with Rastrigin function, iterations={iters}")
        modified_swarm = ModifiedSwarm(num_particles=30, objective_fn=Rastrigin_function)
        modified_pso_simulation = modified_swarm.simulate()
        for _ in range(iters):
            swarm_state, g_best = next(modified_pso_simulation)
        print(f"  -> Best Solution: {g_best}, Fitness: {modified_swarm.fitness(g_best)}")

    # Test modified PSO with Styblinski-Tang function
    for iters in iterations:
        print(f"\nTesting Modified PSO with Styblinski-Tang function, iterations={iters}")
        modified_swarm = ModifiedSwarm(num_particles=30, objective_fn=Styblinski_Tang_function)
        modified_pso_simulation = modified_swarm.simulate()
        for _ in range(iters):
            swarm_state, g_best = next(modified_pso_simulation)
        print(f"  -> Best Solution: {g_best}, Fitness: {modified_swarm.fitness(g_best)}")


Testing Modified PSO with Rastrigin function, iterations=5
  -> Best Solution: [0.02511449941003585, -0.0153815327219613], Fitness: 2.152310929553309

Testing Modified PSO with Rastrigin function, iterations=10
  -> Best Solution: [0.029407077935977632, -0.07766230456812695], Fitness: 3.4407417538834544

Testing Modified PSO with Rastrigin function, iterations=50
  -> Best Solution: [0.05632496421734851, -0.015877810420100613], Fitness: 2.5919704860234347

Testing Modified PSO with Styblinski-Tang function, iterations=5
  -> Best Solution: [-0.1215931045387082, -0.2996277856856633], Fitness: -1.8854065355844134

Testing Modified PSO with Styblinski-Tang function, iterations=10
  -> Best Solution: [-0.12119086725617506, -0.09982001985599631], Fitness: -0.7495798208246527

Testing Modified PSO with Styblinski-Tang function, iterations=50
  -> Best Solution: [0.004481422741784091, 0.03690843826930651], Fitness: 0.09241705284635643


In [ ]:
# Result using modified ABC algorithm

iterations = [5, 10, 50]



3.	Give the results for the best estimated minimal iterations and population. Give the average results over this iterations and population. (Meaning you run this a few times with this number of population and iterations, and get the average results). You may also give the calculation time with the computer specifications. Put all the results in a tabular format (i.e. table form).


4.	Comment on the results that you have obtained, e.g. which algorithm is probably better, etc. and whether the modifications you introduced have made the optimization better or worse. Explain why it succeeded or failed. You may need to perform more experiments to justify you answers.